# Set Metrics: Precision, Recall, MAP, and MRR as Estimators

This narrative notebook imports the canonical, tested reference implementation
`set_metrics_precision_recall_map_mrr.py` and walks the topic section by section. The `.py` **owns
every number**; here we just call it so the claims render as executed output.

The published retrieval stack *measures* recall@k everywhere but never *defines* it. This topic
defines the whole set-metric family — precision, recall, the precision–recall curve, Average
Precision, MAP, and MRR — over the **same three legs** (BM25, the dense dual encoder, late-interaction
MaxSim) scored against the **same neutral** `brute_topk` MaxSim ground truth the capstone used, and
then reframes every metric as an **estimator with variance**.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks" / "set-metrics-precision-recall-map-mrr"))
import numpy as np
import set_metrics_precision_recall_map_mrr as sm

c = sm._corpus()
print("docs", c["docs"].shape, " queries", c["queries"].shape, " legs", sm.LEG_NAMES)
print("|R| (set regime) =", c["r_size"], "  |R| (known-item) = 1   queries =", c["n_queries"])

## Movement 0 — one shared corpus, a neutral truth, two qrel regimes, three full-view legs

The corpus is the PLAID token cloud. The ground truth is **exact MaxSim** (`brute_topk`) — a *neutral*
oracle: no candidate-generation leg *is* the oracle, so the three legs genuinely differ in quality. We
read the one truth two ways: the **set regime** ($|R| = 10$, the top-$k$ MaxSim neighbours) and the
**known-item regime** ($|R| = 1$, the single top-1 doc).

In [ ]:
q = 0
print("qrels_set[0]  (|R| = 10):", sorted(c["qrels_set"][q]))
print("qrels_ki[0]   (|R| =  1):", sorted(c["qrels_ki"][q]))
print("dense leg top-10 for q0 :", sm._ranking(c, "dense", q)[:10])

## Movement 1 — set metrics at a cutoff: precision and recall

For a ranking and a relevant set $R$, $P@k = |{\rm top}_k \cap R| / k$ (the *purity* of the cutoff)
and $R@k = |{\rm top}_k \cap R| / |R|$ (the *coverage* of $R$). As we slide the cutoff $k$ up, recall
is **non-decreasing** but precision **wobbles** — the headline asymmetry.

In [ ]:
wq = sm.pick_worked_query(c, "dense")
r = sm._ranking(c, "dense", wq)
R = c["qrels_set"][wq]
print(f"worked query {wq}, dense leg, relevant ranks: {sm.relevant_ranks(r, R)}")
for k in (1, 3, 5, 10, 25):
    print(f"  k={k:2d}   P@k={sm.precision_at_k(r, R, k):.3f}   R@k={sm.recall_at_k(r, R, k):.3f}")
print("recall@N =", sm.recall_at_k(r, R, c["n_docs"]), "(every relevant doc is somewhere in a full ranking)")

## Movement 2 — the precision–recall curve and Average Precision

The PR curve is the sawtooth of $(R@k, P@k)$ at each relevant hit. **Average Precision is the area
under it**: each relevant hit advances recall by exactly $1/|R|$, so

$$\mathrm{AP} \;=\; \sum_i (R_i - R_{i-1})\,P_i \;=\; \frac{1}{|R|}\sum_{i} \frac{i}{\mathrm{pos}_i},$$

the mean of precision-at-each-relevant-rank. We verify the two forms agree to floating point. The
**interpolated** curve (the monotone upper envelope) always has area $\ge$ the raw AP — interpolation
is a *convention that inflates* the number, not a more correct AP.

In [ ]:
ap = sm.average_precision(r, R)
ap_area = sm.ap_via_pr_area(r, R)
iap = sm.interpolated_ap(r, R)
print(f"AP (relevant-ranks mean) = {ap:.6f}")
print(f"AP (area under PR curve) = {ap_area:.6f}   identical? {abs(ap - ap_area) < 1e-12}")
print(f"interpolated AP          = {iap:.6f}   (>= raw AP? {iap >= ap - 1e-12})")
print("PR-curve corners (recall, precision):",
      [(round(x, 3), round(y, 3)) for x, y in sm.pr_curve(r, R)])

## Movement 3 — aggregation: MAP, MRR, and the MAP $=$ MRR identity

MAP and MRR are sample means across queries: $\mathrm{MAP} = \tfrac1Q\sum_q \mathrm{AP}_q$ and
$\mathrm{MRR} = \tfrac1Q\sum_q \mathrm{RR}_q$ with $\mathrm{RR} = 1/(\text{rank of first relevant})$.
**Collapse anchor:** with exactly one relevant doc at rank $r$, $\mathrm{AP} = \tfrac11\cdot\tfrac1r =
\tfrac1r = \mathrm{RR}$, so in the known-item regime $\mathrm{MAP} = \mathrm{MRR}$ exactly.

In [ ]:
for leg in sm.LEG_NAMES:
    mapv = sm.mean_average_precision(c, leg, "qrels_set")
    mrrv = sm.mean_reciprocal_rank(c, leg, "qrels_ki")
    map_ki = sm.mean_average_precision(c, leg, "qrels_ki")     # AP in the known-item regime
    print(f"{leg:16s} MAP(set)={mapv:.4f}   MRR(known-item)={mrrv:.4f}   "
          f"MAP(known-item)={map_ki:.4f}  ==MRR? {abs(map_ki - mrrv) < 1e-12}")

## Movement 4 — metrics as ESTIMATORS: standard error and the $1/\sqrt{n}$ law

A reported MAP is a *sample mean*, hence a point estimate with standard error $\mathrm{SE} =
\hat\sigma/\sqrt{n}$. Resampling the per-query AP at growing $n$ shows the empirical SE tracking
$\hat\sigma/\sqrt n$ (so $\mathrm{SE}\cdot\sqrt n$ is ~constant). And two systems whose CIs overlap
**cannot be distinguished** at that query count — the hook for significance testing.

In [ ]:
for leg in sm.LEG_NAMES:
    s = sm.metric_summary(sm.per_query_ap(c, leg))
    print(f"{leg:16s} MAP={s['mean']:.4f}  std={s['std']:.4f}  SE={s['se']:.4f}  "
          f"95% CI=[{s['ci_lo']:.4f}, {s['ci_hi']:.4f}]")
print("\nSE ~ 1/sqrt(n) (dense):")
for row in sm.se_scaling_experiment(c, "dense", seed=1):
    print(f"  n={row['n']:3d}  empirical SE={row['empirical_se']:.4f}  theory={row['theory_se']:.4f}  "
          f"SE*sqrt(n)={row['se_root_n']:.4f}")
sep = sm.projected_separation_n(c, "dense", "late_interaction")
print(f"\ndense vs late_interaction: projected CIs overlap at n=5 "
      f"({sm.projected_overlap(c, 'dense', 'late_interaction', 5)}); they separate at n={sep}.")

## The headline flip — metric choice changes the verdict

MAP rewards ranking the *whole* relevant set high; MRR rewards getting *one* answer to rank 1. So the
verdict can **reverse** between metrics. We detect a pairwise reversal (run, never assumed).

In [ ]:
flip = sm.metric_choice_flips_verdict(c)
for lg, v in flip["table"].items():
    print(f"  {lg:16s} MAP={v['map']:.3f}   MRR={v['mrr']:.3f}")
print("reversal:", flip["reversal"], " flips:", flip["flips"])

## Everything verified

Re-run the full assert harness (every pedagogical claim the topic makes is an `assert`).

In [ ]:
sm._run_all()